# 09F · SpotClean on a score-defined native window

Open this add-on beside 09E in the **same Python kernel** and choose **Run All**. The first stage finds R 4.5, installs SpotClean in a user library if needed, and tests it on a synthetic count matrix. The second stage fits one contiguous window of native 2 µm squares. Results include wall time, peak R memory, marker retention and a return-to-chat ZIP.

The tissue measurement is the stored, normalized `he_tissue_score` from combined eosin and hematoxylin intensity. The default pilot treats low-score squares outside Proseg masks as candidate background. The notebook carries `qc_class` alongside the score for comparison. No slide image or image reader is required.

In [ ]:
from pathlib import Path
from IPython.display import display, Image, Markdown
import json
import pandas as pd
from bins09d import STUDY_ROOT, TEMP_PROJECT_ROOT
from review09 import show_review_zip
from run09f import run_09f

RESULT_ROOT = Path("09_pilot_results")
RUN_DIR = None        # Optional completed 09B directory
PATH_OVERRIDES = {}   # 09D keys, including "zarr" or "aligned_qc" for moved inputs
ZARR_TABLE = "square_002um"
RSCRIPT = None        # Optional absolute path to R 4.5.2's Rscript
PAIRED_FITS = None    # Auto-detect the matching 09E paired_fits.json if available

SIDE_BINS = 40        # 80 µm across, 1,600 positions including the halo
HALO_BINS = 4         # 8 µm excluded on each edge for the cell-level readout
SCORE_MAX = 0.10      # Pilot threshold on the stored 0–1 staining score
MIN_BACKGROUND_FRACTION = 0.25
REQUIRE_SR_BACKGROUND = False   # True restricts candidate background to outside Space Ranger tissue
ORIGIN_RC = None      # Optional (minimum array_row, minimum array_col) to repeat an exact window
FIT_MINUTES = 30      # Wall-time limit for the biological R fit; installation timed separately

## Installation and pilot

The selection uses staining, mask membership and position before reading RNA. It first searches for low-score squares outside both Proseg and Space Ranger tissue. If none of the candidate windows have sufficient background of that kind, the default also considers low-score, unassigned squares within Space Ranger tissue. The receipt records which definition was used. Spots meeting the background rule have zero native expression **by assumption in SpotClean**. A missing score never qualifies a square as background.

The installer reuses compatible installed dependencies. First-time setup requires access to package repositories and may compile R packages. Sites that use internal package mirrors can set `CT26_CRAN_REPO` and `CT26_BIOC_MIRROR` before running. The full stage logs stay in the result folder.

In [ ]:
out, archive, status = run_09f(
    result_root=RESULT_ROOT, run_dir=RUN_DIR, study_root=STUDY_ROOT,
    temp_project_root=TEMP_PROJECT_ROOT, overrides=PATH_OVERRIDES, zarr_table=ZARR_TABLE,
    rscript=RSCRIPT, paired_fits=PAIRED_FITS, side_bins=SIDE_BINS,
    halo_bins=HALO_BINS, score_max=SCORE_MAX, origin_rc=ORIGIN_RC,
    min_background_fraction=MIN_BACKGROUND_FRACTION, require_sr_background=REQUIRE_SR_BACKGROUND,
    fit_minutes=FIT_MINUTES,
)
print("Status", status["state"], " |  Results", out)
display(pd.DataFrame(status["stages"]).T.fillna(""))
review_download = show_review_zip(archive)

## Tissue scores and computational cost

The figure uses the native capture grid. Each pixel represents a measured position. The score distribution table covers the search area and is also available if no window qualifies. A first-time package build and smoke test can dominate setup time. The pilot records R-process memory separately from Python data preparation. Runtime changes with tissue fraction, expression and region size.

In [ ]:
inputs, fit, summary = out / "input", out / "fit", out / "summary"
if (inputs / "score_search.csv").is_file():
    display(pd.read_csv(inputs / "score_search.csv").round(3))
if (inputs / "roi_receipt.json").is_file():
    roi = json.loads((inputs / "roi_receipt.json").read_text())
    display(pd.Series({k: roi.get(k) for k in ("origin_rc", "n_positions", "n_background",
        "background_mode", "background_classes", "score_missing", "n_zero_positions", "n_complete_cells")}))
if (summary / "score_and_roles.png").is_file():
    display(Image(filename=str(summary / "score_and_roles.png"), width=780))
if (fit / "fit.tsv").is_file():
    display(pd.read_csv(fit / "fit.tsv", sep="\t").T)

## Spillover and cell boundaries

The myeloid comparison asks how much marker signal survives in a labeled source cell and in nearby pan-T-supported cells. The denominator is the observed RNA on the same complete hard-mask footprint. Very small groups and low raw counts limit inference. The optional historical SPARKLE comparison requires matching input support on shared cells.

The coreceptor table measures raw `Cd4`, `Cd8a` and `Cd8b1` both inside and outside masks across the continuous-score strata. Outside-mask counts on high-scoring tissue motivate a look at segmentation. These counts alone cannot identify their cell of origin.

In [ ]:
if (summary / "marker_readout.md").is_file():
    display(Markdown((summary / "marker_readout.md").read_text()))
if (summary / "marker_readout.png").is_file():
    display(Image(filename=str(summary / "marker_readout.png"), width=980))
if (summary / "paired_sparkle.csv").is_file():
    matched = pd.read_csv(summary / "paired_sparkle.csv")
    if len(matched):
        display(matched)
print("Return ZIP", review_download)